# scikit-learn Synthetic Dataset Generators

scikit-learn provides built-in functions for generating synthetic datasets. These are useful for algorithm benchmarking and controlled experiments — they are not intended as substitutes for realistic data synthesis.

| Function | Task type | Key parameters |
|---|---|---|
| `make_classification` | Classification | `n_classes`, `class_sep`, `weights`, `n_informative` |
| `make_regression` | Regression | `n_features`, `noise` |
| `make_blobs` | Clustering | `centers`, `cluster_std` |
| `make_moons` | Non-linear classification | `noise` |
| `make_circles` | Non-linear classification | `factor`, `noise` |
| `make_multilabel_classification` | Multi-label | `n_classes`, `n_labels` |
| `make_friedman1/2/3` | Non-linear regression | `n_features`, `noise` |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import (
    make_classification, make_regression, make_blobs,
    make_moons, make_circles, make_multilabel_classification,
    make_friedman1
)

plt.rcParams['figure.figsize'] = (6, 4)
COLOURS = plt.rcParams['axes.prop_cycle'].by_key()['color']

---
## 1. `make_classification`

Generates an n-class classification problem with controllable class separation, feature redundancy, and class imbalance. Useful for testing classifiers under realistic conditions.

In [ ]:
X, y = make_classification(
    n_samples=300, n_features=2, n_informative=2, n_redundant=0,
    n_classes=2, class_sep=1.5, weights=[0.3, 0.7],
    flip_y=0, n_clusters_per_class=1, random_state=42
)

for cls in [0, 1]:
    plt.scatter(X[y == cls, 0], X[y == cls, 1],
                c=COLOURS[cls], alpha=0.6, s=20, label=f'class {cls}')
plt.title('make_classification (2 features, imbalanced)')
plt.legend()
plt.tight_layout()
plt.show()

print(f'Shape: {X.shape}   Class counts: {dict(zip(*np.unique(y, return_counts=True)))}')

---
## 2. `make_regression`

Generates a regression problem where the target is a linear combination of the input features plus Gaussian noise. The `coef` flag returns the true coefficients for validation.

In [ ]:
X, y, coef = make_regression(
    n_samples=200, n_features=1, noise=20, coef=True, random_state=42
)

x_line = np.linspace(X.min(), X.max(), 100)
y_line = coef * x_line

plt.scatter(X, y, alpha=0.5, s=20, label='samples')
plt.plot(x_line, y_line, color='red', linewidth=2, label=f'true fit (coef={coef:.1f})')
plt.title('make_regression (1 feature, noise=20)')
plt.legend()
plt.tight_layout()
plt.show()

print(f'Shape: {X.shape}   True coefficient: {coef:.3f}')

---
## 3. `make_blobs`

Generates isotropic Gaussian blobs. The simplest clustering benchmark — each blob is a well-separated spherical cluster.

In [ ]:
X, y = make_blobs(n_samples=300, centers=4, n_features=2, cluster_std=1.2, random_state=42)

for cls in range(4):
    plt.scatter(X[y == cls, 0], X[y == cls, 1],
                c=COLOURS[cls], alpha=0.6, s=20, label=f'cluster {cls}')
plt.title('make_blobs (4 clusters)')
plt.legend()
plt.tight_layout()
plt.show()

print(f'Shape: {X.shape}   Cluster counts: {dict(zip(*np.unique(y, return_counts=True)))}')

---
## 4. `make_moons`

Generates two interleaving half-circles — a classic non-linearly separable dataset. Useful for testing kernel SVMs, neural networks, and other methods that can learn non-linear boundaries.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))

for ax, noise in zip(axes, [0.0, 0.1, 0.3]):
    X, y = make_moons(n_samples=300, noise=noise, random_state=42)
    for cls in [0, 1]:
        ax.scatter(X[y == cls, 0], X[y == cls, 1],
                   c=COLOURS[cls], alpha=0.6, s=15)
    ax.set_title(f'noise={noise}')

fig.suptitle('make_moons at increasing noise levels')
plt.tight_layout()
plt.show()

---
## 5. `make_circles`

Generates a large circle containing a smaller circle — another non-linearly separable benchmark. `factor` controls the ratio of the inner to outer circle radius.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))

for ax, factor in zip(axes, [0.3, 0.5, 0.8]):
    X, y = make_circles(n_samples=300, factor=factor, noise=0.05, random_state=42)
    for cls in [0, 1]:
        ax.scatter(X[y == cls, 0], X[y == cls, 1],
                   c=COLOURS[cls], alpha=0.6, s=15)
    ax.set_title(f'factor={factor}')
    ax.set_aspect('equal')

fig.suptitle('make_circles at different radius ratios')
plt.tight_layout()
plt.show()

---
## 6. `make_multilabel_classification`

Generates a multi-label classification problem where each sample can belong to several classes simultaneously. The label matrix `y` has shape `(n_samples, n_classes)` with binary entries.

In [ ]:
X, y = make_multilabel_classification(
    n_samples=200, n_features=10, n_classes=4, n_labels=2, random_state=42
)

label_counts = y.sum(axis=0)

plt.bar(range(len(label_counts)), label_counts, color=COLOURS[:len(label_counts)])
plt.xticks(range(len(label_counts)), [f'label {i}' for i in range(len(label_counts))])
plt.ylabel('Count')
plt.title('make_multilabel_classification — label frequency')
plt.tight_layout()
plt.show()

print(f'X shape: {X.shape}   y shape: {y.shape}')
print(f'Label frequencies: {y.mean(axis=0).round(2)}')
print(f'Avg labels per sample: {y.sum(axis=1).mean():.2f}')

---
## 7. `make_friedman1`

Generates the Friedman \#1 non-linear regression benchmark. The target is:

$$y = 10 \sin(\pi x_1 x_2) + 20(x_3 - 0.5)^2 + 10 x_4 + 5 x_5 + \epsilon$$

Only the first 5 features are informative; the rest are noise. Useful for testing feature selection and non-linear regression methods.

In [ ]:
X, y = make_friedman1(n_samples=500, n_features=10, noise=1.0, random_state=42)

correlations = [abs(np.corrcoef(X[:, i], y)[0, 1]) for i in range(X.shape[1])]
colours = [COLOURS[0] if i < 5 else '#bbbbbb' for i in range(len(correlations))]

plt.bar(range(len(correlations)), correlations, color=colours)
plt.xticks(range(len(correlations)), [f'x{i+1}' for i in range(len(correlations))])
plt.ylabel('|Correlation with y|')
plt.title('make_friedman1 — feature relevance (blue = informative, grey = noise)')
plt.tight_layout()
plt.show()

print(f'Shape: {X.shape}   y range: [{y.min():.1f}, {y.max():.1f}]')

---
## When to Use scikit-learn Generators vs Realistic Synthesis Methods

| Use case | Recommended approach |
|---|---|
| Algorithm benchmarking, controlled experiments | scikit-learn generators (this notebook) |
| Realistic tabular data with known distributions | Monte Carlo, Cholesky (see `notebooks/`) |
| Realistic tabular data with known joint distribution | Copula (see `notebooks/ch13_copula/`) |
| Privacy-preserving surrogate for real data | GAN (see `notebooks/ch12_gan/`) |
| Merging partial datasets | ProbCon (see `notebooks/ch11_probcon/`) |